# Late Delivery Risk Prediction Preprocessing Pipeline

Improved preprocessing pipeline with leakage prevention, controlled vocabulary handling, feature engineering, multicollinearity reduction, and MLflow tracking.

In [1]:
!pip  install mlflow -q
!pip install dagshub -q
!pip install category_encoders -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.4/49.4 kB 3.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 3.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 32.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 34.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 26.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.0/212.0 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.9/94.9 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 936.9/936.9 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 214

# Imports

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import shutil

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from statsmodels.stats.outliers_influence import variance_inflation_factor

from category_encoders import TargetEncoder

import os
import json
import joblib
import mlflow
import dagshub
from google.colab import drive

pd.set_option("display.max_columns", None)


In [3]:


drive.mount("/content/drive")

dir_path = "/content/drive/MyDrive/um/late_delivery_prediction"
os.chdir(dir_path)
print(os.getcwd())


Mounted at /content/drive
/content/drive/MyDrive/um/late_delivery_prediction


# MLflow Experiment Tracking

In [4]:
DAGSHUB_USERNAME = "sharifa-mohamed"
REPO_NAME = "late_delivery_prediction"
experiment_name = "late_delivery_prediction"


dagshub.init(repo_owner=DAGSHUB_USERNAME, repo_name=REPO_NAME, mlflow=True)

print("\nConnected MLflow Tracking URI:")
print(mlflow.get_tracking_uri())

mlflow.set_experiment(experiment_name)

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=f852a254-f9ed-45c7-945c-1255cd20e7e8&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=985e4e2c5e13abba0edb6e0e8d111a6912f9364b5c9c7d7abf0fd46e8444bee1




Accessing as sharifa-mohamed

Initialized MLflow to track repo "sharifa-mohamed/late_delivery_prediction"

Repository sharifa-mohamed/late_delivery_prediction initialized!


Connected MLflow Tracking URI:
https://dagshub.com/sharifa-mohamed/late_delivery_prediction.mlflow


<Experiment: artifact_location='mlflow-artifacts:/8a7aa0333d8e43219410be10c0d25f4b', creation_time=1780407476578, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1780407476578, lifecycle_stage='active', name='late_delivery_prediction', tags={'mlflow.experimentKind': 'custom_model_development'}, trace_location=None, workspace='default'>

# Load Dataset

In [5]:
file_path = "data/APL_Logistics.csv"

raw_df = pd.read_csv(file_path, encoding="cp1252")

print("\nDataset Loaded Successfully")
print(f"Dataset Shape: {raw_df.shape}")


Dataset Loaded Successfully
Dataset Shape: (180519, 40)


# Initial Dataset Inspection

In [6]:
print("\nDataset Info:")
raw_df.info()

print("\nColumn Data Types:")
print(raw_df.dtypes.value_counts())

print("\nFirst 5 Rows:")
print(raw_df.head())

print("\nNumerical Summary:")
display(raw_df.describe().T.round(2))

print("\nCategorical Summary:")
display(raw_df.describe(include="object").T.round(2))


Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 180519 entries, 0 to 180518
Data columns (total 40 columns):
 #   Column                         Non-Null Count   Dtype  
---  ------                         --------------   -----  
 0   Type                           180519 non-null  object 
 1   Days for shipping (real)       180519 non-null  int64  
 2   Days for shipment (scheduled)  180519 non-null  int64  
 3   Benefit per order              180519 non-null  float64
 4   Sales per customer             180519 non-null  float64
 5   Delivery Status                180519 non-null  object 
 6   Late_delivery_risk             180519 non-null  int64  
 7   Category Id                    180519 non-null  int64  
 8   Category Name                  180519 non-null  object 
 9   Customer City                  180519 non-null  object 
 10  Customer Country               180519 non-null  object 
 11  Customer Fname                 180519 non-null  object 
 12  Customer Id    

,count,mean,std,min,25%,50%,75%,max
Days for shipping (real),180519.0,3.50,1.62,0.00,2.00,3.00,5.00,6.00
Days for shipment (scheduled),180519.0,2.93,1.37,0.00,2.00,4.00,4.00,4.00
Benefit per order,180519.0,21.97,104.43,-4274.98,7.00,31.52,64.80,911.80
Sales per customer,180519.0,183.11,120.04,7.49,104.38,163.99,247.40,1939.99
Late_delivery_risk,180519.0,0.55,0.50,0.00,0.00,1.00,1.00,1.00
Category Id,180519.0,31.85,15.64,2.00,18.00,29.00,45.00,76.00
Customer Id,180519.0,6691.38,4162.92,1.00,3258.50,6457.00,9779.00,20757.00
Customer Zipcode,180516.0,35921.13,37542.46,603.00,725.00,19380.00,78207.00,99205.00
Department Id,180519.0,5.44,1.63,2.00,4.00,5.00,7.00,12.00
Latitude,180519.0,29.72,9.81,-33.94,18.27,33.14,39.28,48.78



Categorical Summary:


,count,unique,top,freq
Type,180519,4,DEBIT,69295
Delivery Status,180519,4,Late delivery,98977
Category Name,180519,50,Cleats,24551
Customer City,180519,563,Caguas,66770
Customer Country,180519,2,EE. UU.,111146
Customer Fname,180519,782,Mary,65150
Customer Lname,180511,1109,Smith,64104
Customer Segment,180519,3,Consumer,93504
Customer State,180519,46,PR,69373
Customer Street,180519,7458,9126 Wishing Expressway,122


# Duplicate Row Check

In [7]:
duplicate_count = raw_df.duplicated().sum()

print(f"\nDuplicate Rows Found: {duplicate_count}")


Duplicate Rows Found: 0


# Remove high cordinality columns

In [8]:
for col in raw_df.select_dtypes(include=["object", "string"]).columns.tolist():
    print(col, raw_df[col].nunique())

Type 4
Delivery Status 4
Category Name 50
Customer City 563
Customer Country 2
Customer Fname 782
Customer Lname 1109
Customer Segment 3
Customer State 46
Customer Street 7458
Department Name 11
Market 5
Order City 3597
Order Country 164
Order Region 23
Order State 1089
Order Status 9
Product Name 118
Shipping Mode 4


In [9]:
high_cardinality_cols = [
    "Customer Street",
    "Customer Lname",
    "Customer Fname",
    "Order City",
    "Order State"


]

clean_df = raw_df.drop(columns=high_cardinality_cols)

print("\nShape After Removing high cordinality Columns:")
print(clean_df.shape)


Shape After Removing high cordinality Columns:
(180519, 35)


# Leakage Inspection

In [10]:
print("\nOrder Status Distribution:")
print(clean_df["Order Status"].value_counts(dropna=False))

print("\nDelivery Status Distribution:")
print(clean_df["Delivery Status"].value_counts(dropna=False))

print("\nLate Delivery Risk Distribution:")
print(clean_df["Late_delivery_risk"].value_counts(dropna=False))

print("\nDelivery Status vs Late Delivery Risk Crosstab:")
print(pd.crosstab(clean_df["Delivery Status"], clean_df["Late_delivery_risk"]))


Order Status Distribution:
Order Status
COMPLETE           59491
PENDING_PAYMENT    39832
PROCESSING         21902
PENDING            20227
CLOSED             19616
ON_HOLD             9804
SUSPECTED_FRAUD     4062
CANCELED            3692
PAYMENT_REVIEW      1893
Name: count, dtype: int64

Delivery Status Distribution:
Delivery Status
Late delivery        98977
Advance shipping     41592
Shipping on time     32196
Shipping canceled     7754
Name: count, dtype: int64

Late Delivery Risk Distribution:
Late_delivery_risk
1    98977
0    81542
Name: count, dtype: int64

Delivery Status vs Late Delivery Risk Crosstab:
Late_delivery_risk      0      1
Delivery Status                 
Advance shipping    41592      0
Late delivery           0  98977
Shipping canceled    7754      0
Shipping on time    32196      0


# Remove Leakage Features

In [11]:
future_leakage_columns = [
    "Delivery Status",
    "Days for shipping (real)",
    "Benefit per order",
    "Order Profit Per Order",
    "Order Item Profit Ratio",
    "Order Status"
]

clean_df.drop(columns=future_leakage_columns, inplace=True, errors="ignore")

print("\nShape After Leakage Removal:")
print(clean_df.shape)


Shape After Leakage Removal:
(180519, 29)


# Consistency Validation

In [12]:
print("\nRows Where Scheduled Shipping Days == 0:")

print(
    clean_df[
        clean_df["Days for shipment (scheduled)"] == 0
    ]["Shipping Mode"].value_counts(dropna=False)
)

print("\nChecking Sales per customer vs Order Item Total:")

print((clean_df["Sales per customer"] == clean_df["Order Item Total"]).all())

print("\nChecking Product Price vs Order Item Product Price:")

print((clean_df["Order Item Product Price"] == clean_df["Product Price"]).all())


Rows Where Scheduled Shipping Days == 0:
Shipping Mode
Same Day    9737
Name: count, dtype: int64

Checking Sales per customer vs Order Item Total:
True

Checking Product Price vs Order Item Product Price:
True


# Remove Redundant Features

In [13]:
redundant_columns = [
    "Sales per customer",
    "Product Price"
]

clean_df.drop(columns=redundant_columns, inplace=True, errors="ignore")

print("\nShape After Removing Redundant Features:")
print(clean_df.shape)


Shape After Removing Redundant Features:
(180519, 27)


# Train Test Split

In [14]:
X = clean_df.drop(columns=["Late_delivery_risk"])
y = clean_df["Late_delivery_risk"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

X_train_raw = X_train.copy()
X_test_raw = X_test.copy()

X_train = X_train.copy()
X_test = X_test.copy()


print("\nTrain Test Split Complete")

print(f"X_train Shape: {X_train.shape}")
print(f"X_test Shape: {X_test.shape}")


Train Test Split Complete
X_train Shape: (144415, 26)
X_test Shape: (36104, 26)


# Handling missing values

In [15]:

def analyze_missing_values(df, dataset_name="Dataset"):
    missing_count = df.isnull().sum()
    missing_percent = (df.isnull().sum() / len(df)) * 100

    missing_summary = pd.concat([missing_count, missing_percent], axis=1, keys=['Missing Count', 'Percentage (%)'])
    missing_summary = missing_summary[missing_summary['Missing Count'] > 0].sort_values('Percentage (%)', ascending=False)

    print(f"\nMissing Data Analysis for {dataset_name}:")
    print(missing_summary)

analyze_missing_values(X_train, "Training Data")
analyze_missing_values(X_test, "Testing Data")



Missing Data Analysis for Training Data:
                  Missing Count  Percentage (%)
Customer Zipcode              3        0.002077

Missing Data Analysis for Testing Data:
Empty DataFrame
Columns: [Missing Count, Percentage (%)]
Index: []


In [16]:
# We define a mapping for expected missing values based on columns
imputation_values = {
   "Customer Zipcode": 0.0
}

# Apply imputation to X_train and X_test BEFORE encoding
# Using .fillna() avoids dropping data and handles unseen missing values in production
X_train = X_train.fillna(value=imputation_values)
X_test = X_test.fillna(value=imputation_values)

print("Imputation Complete. Verifying missing values remaining in X_train:")
print(X_train[["Customer Zipcode"]].isnull().sum())

Imputation Complete. Verifying missing values remaining in X_train:
Customer Zipcode    0
dtype: int64


# Feature Engineering

In [17]:
# =====================================================================
# 1. Shipping Pressure Index
# =====================================================================
X_train["shipping_pressure_index"] = np.where(
    X_train["Days for shipment (scheduled)"] == 0,
    X_train["Order Item Quantity"],
    X_train["Order Item Quantity"] / X_train["Days for shipment (scheduled)"]
)

X_test["shipping_pressure_index"] = np.where(
    X_test["Days for shipment (scheduled)"] == 0,
    X_test["Order Item Quantity"],
    X_test["Order Item Quantity"] / X_test["Days for shipment (scheduled)"]
)

# =====================================================================
# 2. Mode Risk Flag
# =====================================================================
high_urgency_modes = ["Same Day", "First Class"]

X_train["is_high_urgency_mode"] = (
    X_train["Shipping Mode"].isin(high_urgency_modes).astype(int)
)

X_test["is_high_urgency_mode"] = (
    X_test["Shipping Mode"].isin(high_urgency_modes).astype(int)
)

# =====================================================================
# 3. Regional Congestion Features
# =====================================================================
regional_volume_shares = (
    X_train["Order Region"].value_counts() / len(X_train)
).to_dict()

X_train["regional_congestion_score"] = (
    X_train["Order Region"].map(regional_volume_shares)
)

global_region_mean = np.mean(list(regional_volume_shares.values()))

X_test["regional_congestion_score"] = (
    X_test["Order Region"]
    .map(regional_volume_shares)
    .fillna(global_region_mean)
)

# =====================================================================
# 4. Bulk Order Indicator
# =====================================================================
X_train["is_bulk_order"] = (
    X_train["Order Item Quantity"] > 3
).astype(int)

X_test["is_bulk_order"] = (
    X_test["Order Item Quantity"] > 3
).astype(int)

# =====================================================================
# 5. Order Complexity Score
# =====================================================================
X_train["order_complexity_score"] = (
    (
        X_train["Order Item Quantity"]
        * X_train["Order Item Product Price"]
    )
    * (1.0 - X_train["Order Item Discount Rate"])
)

X_test["order_complexity_score"] = (
    (
        X_test["Order Item Quantity"]
        * X_test["Order Item Product Price"]
    )
    * (1.0 - X_test["Order Item Discount Rate"])
)

# =====================================================================
# 6. Monetary Stress Features
# =====================================================================
X_train["discount_per_item"] = (
    X_train["Order Item Discount"] /
    (X_train["Order Item Quantity"] + 1)
)

X_test["discount_per_item"] = (
    X_test["Order Item Discount"] /
    (X_test["Order Item Quantity"] + 1)
)

X_train["high_value_order"] = (
    X_train["Order Item Product Price"] *
    X_train["Order Item Quantity"]
)

X_test["high_value_order"] = (
    X_test["Order Item Product Price"] *
    X_test["Order Item Quantity"]
)


# =====================================================================
# 7. Customer Level Features (LEAK-PROOF MAPPING)
# =====================================================================
# Compute aggregates on Train only
cust_counts_map = X_train.groupby("Order Customer Id")["Order Item Quantity"].count()
cust_sums_map   = X_train.groupby("Order Customer Id")["Order Item Quantity"].sum()
cust_sales_map  = X_train.groupby("Order Customer Id")["Sales"].mean()
cust_disc_map   = X_train.groupby("Order Customer Id")["Order Item Discount"].mean()

# Global defaults derived strictly from Train
global_cust_count = cust_counts_map.mean()
global_cust_sum   = cust_sums_map.mean()
global_cust_sales = cust_sales_map.mean()
global_cust_disc  = cust_disc_map.mean()

# Map onto Train
X_train["customer_order_count"]    = X_train["Order Customer Id"].map(cust_counts_map)
X_train["customer_total_quantity"] = X_train["Order Customer Id"].map(cust_sums_map)
X_train["customer_avg_order_value"] = X_train["Order Customer Id"].map(cust_sales_map)
X_train["customer_avg_discount"]    = X_train["Order Customer Id"].map(cust_disc_map)

# Map onto Test and handle unseen customers using Train defaults
X_test["customer_order_count"]     = X_test["Order Customer Id"].map(cust_counts_map).fillna(global_cust_count)
X_test["customer_total_quantity"]  = X_test["Order Customer Id"].map(cust_sums_map).fillna(global_cust_sum)
X_test["customer_avg_order_value"] = X_test["Order Customer Id"].map(cust_sales_map).fillna(global_cust_sales)
X_test["customer_avg_discount"]    = X_test["Order Customer Id"].map(cust_disc_map).fillna(global_cust_disc)


# =====================================================================
# 8. Product Level Features
# =====================================================================
# Compute aggregates on Train only
prod_counts_map = X_train.groupby("Product Name")["Order Item Quantity"].count()
prod_sales_map  = X_train.groupby("Product Name")["Sales"].mean()
prod_disc_map   = X_train.groupby("Product Name")["Order Item Discount"].mean()

# Global defaults derived strictly from Train
global_prod_count = prod_counts_map.mean()
global_prod_sales = prod_sales_map.mean()
global_prod_disc  = prod_disc_map.mean()

# Map onto Train
X_train["product_order_frequency"] = X_train["Product Name"].map(prod_counts_map)
X_train["product_avg_sales"]       = X_train["Product Name"].map(prod_sales_map)
X_train["product_avg_discount"]    = X_train["Product Name"].map(prod_disc_map)

# Map onto Test and handle unseen products using Train defaults
X_test["product_order_frequency"]  = X_test["Product Name"].map(prod_counts_map).fillna(global_prod_count)
X_test["product_avg_sales"]        = X_test["Product Name"].map(prod_sales_map).fillna(global_prod_sales)
X_test["product_avg_discount"]     = X_test["Product Name"].map(prod_disc_map).fillna(global_prod_disc)


# =====================================================================
# 9. Region Level Features
# =====================================================================
# Compute aggregates on Train only
region_counts_map = X_train.groupby("Order Region")["Order Item Quantity"].count()
region_sales_map  = X_train.groupby("Order Region")["Sales"].mean()
region_disc_map   = X_train.groupby("Order Region")["Order Item Discount"].mean()

# Global defaults derived strictly from Train
global_region_count = region_counts_map.mean()
global_region_sales = region_sales_map.mean()
global_region_disc  = region_disc_map.mean()

# Map onto Train
X_train["region_order_volume"] = X_train["Order Region"].map(region_counts_map)
X_train["region_avg_sales"]    = X_train["Order Region"].map(region_sales_map)
X_train["region_avg_discount"] = X_train["Order Region"].map(region_disc_map)

# Map onto Test and handle unseen regions using Train defaults
X_test["region_order_volume"]  = X_test["Order Region"].map(region_counts_map).fillna(global_region_count)
X_test["region_avg_sales"]     = X_test["Order Region"].map(region_sales_map).fillna(global_region_sales)
X_test["region_avg_discount"]  = X_test["Order Region"].map(region_disc_map).fillna(global_region_disc)


# =====================================================================
# 10. Department Level Features
# =====================================================================
# Compute aggregates on Train only
dept_sales_map  = X_train.groupby("Department Name")["Sales"].mean()
dept_counts_map = X_train.groupby("Department Name")["Order Item Quantity"].count()
dept_disc_map   = X_train.groupby("Department Name")["Order Item Discount"].mean()

# Global defaults derived strictly from Train
global_dept_sales = dept_sales_map.mean()
global_dept_count = dept_counts_map.mean()
global_dept_disc  = dept_disc_map.mean()

# Map onto Train
X_train["department_avg_sales"]    = X_train["Department Name"].map(dept_sales_map)
X_train["department_order_volume"] = X_train["Department Name"].map(dept_counts_map)
X_train["department_avg_discount"] = X_train["Department Name"].map(dept_disc_map)

# Map onto Test and handle unseen departments using Train defaults
X_test["department_avg_sales"]     = X_test["Department Name"].map(dept_sales_map).fillna(global_dept_sales)
X_test["department_order_volume"]  = X_test["Department Name"].map(dept_counts_map).fillna(global_dept_count)
X_test["department_avg_discount"]  = X_test["Department Name"].map(dept_disc_map).fillna(global_dept_disc)


print("\nFeature Engineering Complete")



Feature Engineering Complete


# Remove Structural Identifier Columns

In [18]:
structural_cols = [
    "Customer Id",
    "Order Customer Id",
    "Category Id",
    "Department Id",
    "Latitude",
    "Longitude"
]


X_train = X_train.drop(columns=structural_cols, errors="ignore")
X_test = X_test.drop(columns=structural_cols, errors="ignore")

print("\nShape After Removing Structural Columns:")
print(X_train.shape)
print(X_test.shape)


Shape After Removing Structural Columns:
(144415, 40)
(36104, 40)


# Outlier handling

In [19]:
# # Outlier Inspection

outlier_cols= ['Sales', 'Order Item Total', 'Order Item Quantity', 'Order Item Product Price', 'Order Item Discount', 'Order Item Discount Rate', 'shipping_pressure_index', 'order_complexity_score', "customer_order_count","customer_avg_order_value","product_order_frequency","region_order_volume"]

print("\nOutlier Inspection Summary")

for col in outlier_cols:

    q1 = X_train[col].quantile(0.25)
    q3 = X_train[col].quantile(0.75)

    iqr = q3 - q1

    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr

    outlier_count = ((X_train[col] < lower_bound) | (X_train[col] > upper_bound)).sum()

    print(f"\nColumn: {col}")
    print(f"Lower Bound: {lower_bound:.2f}")
    print(f"Upper Bound: {upper_bound:.2f}")
    print(f"Outlier Count: {outlier_count}")
    print(f"Outlier Percentage: {(outlier_count / len(X_train)) * 100:.2f}%")


Outlier Inspection Summary

Column: Sales
Lower Bound: -149.97
Upper Bound: 569.90
Outlier Count: 388
Outlier Percentage: 0.27%

Column: Order Item Total
Lower Bound: -110.15
Upper Bound: 461.93
Outlier Count: 1557
Outlier Percentage: 1.08%

Column: Order Item Quantity
Lower Bound: -2.00
Upper Bound: 6.00
Outlier Count: 0
Outlier Percentage: 0.00%

Column: Order Item Product Price
Lower Bound: -174.99
Upper Bound: 424.98
Outlier Count: 1620
Outlier Percentage: 1.12%

Column: Order Item Discount
Lower Bound: -31.25
Upper Bound: 66.75
Outlier Count: 6000
Outlier Percentage: 4.15%

Column: Order Item Discount Rate
Lower Bound: -0.14
Upper Bound: 0.34
Outlier Count: 0
Outlier Percentage: 0.00%

Column: shipping_pressure_index
Lower Bound: -0.88
Upper Bound: 2.12
Outlier Count: 13450
Outlier Percentage: 9.31%

Column: order_complexity_score
Lower Bound: -110.14
Upper Bound: 461.93
Outlier Count: 1557
Outlier Percentage: 1.08%

Column: customer_order_count
Lower Bound: -4.50
Upper Bound: 31

In [20]:
# # Outlier Clipping Using IQR Method

outlier_clip_columns = [
    "Sales",
    "Order Item Total",
    "Order Item Product Price",
    "Order Item Discount",

]

outlier_bounds = {}

for col in outlier_clip_columns:

    q1 = X_train[col].quantile(0.25)
    q3 = X_train[col].quantile(0.75)

    iqr = q3 - q1

    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr

    outlier_bounds[col] = (lower_bound, upper_bound)

    X_train[col] = X_train[col].clip(lower=lower_bound, upper=upper_bound)

    X_test[col] = X_test[col].clip(lower=lower_bound, upper=upper_bound)

print("\nOutlier Clipping Completed")


Outlier Clipping Completed


# Target encode categorical Features

In [21]:
numeric_cols_before_encoding = X_train.select_dtypes(include=np.number).columns.tolist()
print("numeric_cols_before_encoding:", numeric_cols_before_encoding)
categorical_cols = X_train.select_dtypes(include=["object", "string"]).columns.tolist()


print(f"Features to target encode: {categorical_cols}")
print(f"Shape Before Target Encoding - Train: {X_train.shape}, Test: {X_test.shape}")

# 1. Initialize the TargetEncoder
# smoothing: balances the category mean vs the global mean (helps with rare categories)
encoder = TargetEncoder(handle_unknown='value', handle_missing='value', cols=categorical_cols, smoothing=10.0)

# 2. Fit on TRAINING data and transform it (using y_train to learn the risk weights)
X_train_encoded = encoder.fit_transform(X_train, y_train)

# 3. Transform TEST data using the weights learned from the training data
X_test_encoded = encoder.transform(X_test)

print("\nShape After Target Encoding:")
print(f"Train DataFrame Shape: {X_train_encoded.shape}")
print(f"Test DataFrame Shape: {X_test_encoded.shape}")

# Optional: Verify that categorical strings are now numerical risks
print("\nSample of encoded features from X_train:")
print(X_train_encoded[categorical_cols].head())

numeric_cols_before_encoding: ['Days for shipment (scheduled)', 'Customer Zipcode', 'Order Item Discount', 'Order Item Discount Rate', 'Order Item Product Price', 'Order Item Quantity', 'Sales', 'Order Item Total', 'shipping_pressure_index', 'is_high_urgency_mode', 'regional_congestion_score', 'is_bulk_order', 'order_complexity_score', 'discount_per_item', 'high_value_order', 'customer_order_count', 'customer_total_quantity', 'customer_avg_order_value', 'customer_avg_discount', 'product_order_frequency', 'product_avg_sales', 'product_avg_discount', 'region_order_volume', 'region_avg_sales', 'region_avg_discount', 'department_avg_sales', 'department_order_volume', 'department_avg_discount']
Features to target encode: ['Type', 'Category Name', 'Customer City', 'Customer Country', 'Customer Segment', 'Customer State', 'Department Name', 'Market', 'Order Country', 'Order Region', 'Product Name', 'Shipping Mode']
Shape Before Target Encoding - Train: (144415, 40), Test: (36104, 40)

Shape A

# Separate Tree and Logistic Regression Pipelines

In [22]:
X_train_tree = X_train_encoded.copy()
X_test_tree = X_test_encoded.copy()

X_train_lr = X_train_encoded.copy()
X_test_lr = X_test_encoded.copy()

print("\nPipeline Partitioning Complete")


Pipeline Partitioning Complete


# Logistic Regression Feature Scaling

In [23]:
scaling_columns = [
    # 1. Raw Continuous Numerical Features
    "Days for shipment (scheduled)",
    "Order Item Discount",
    "Order Item Discount Rate",
    "Order Item Product Price",
    "Order Item Quantity",
    "Sales",
    "Order Item Total",

    # 2. Engineered Features
    "shipping_pressure_index",
    "regional_congestion_score",
    "order_complexity_score",
    "discount_per_item",
    "high_value_order",
    "customer_order_count",
    "customer_total_quantity",
    "customer_avg_order_value",
    "customer_avg_discount",
    "product_order_frequency",
    "product_avg_sales",
    "product_avg_discount",
    "region_order_volume",
    "region_avg_sales",
    "region_avg_discount",
    "department_avg_sales",
    "department_order_volume",
    "department_avg_discount",

    # 3. Spatial Identifiers (High Variance)
    "Customer Zipcode"
]

existing_scale_columns = [
    col for col in scaling_columns
    if col in X_train_lr.columns
]

scaler = StandardScaler()

X_train_lr_scaled = X_train_lr.copy()
X_test_lr_scaled = X_test_lr.copy()

X_train_lr_scaled[existing_scale_columns] = scaler.fit_transform(
    X_train_lr[existing_scale_columns]
)

X_test_lr_scaled[existing_scale_columns] = scaler.transform(
    X_test_lr[existing_scale_columns]
)

print("\nScaling Complete")


Scaling Complete


In [24]:
display(X_train_lr_scaled.describe().T.round(2))


,count,mean,std,min,25%,50%,75%,max
Type,144415.0,0.55,0.04,0.49,0.49,0.57,0.57,0.58
Days for shipment (scheduled),144415.0,-0.00,1.00,-2.13,-0.68,0.78,0.78,0.78
Category Name,144415.0,0.55,0.01,0.49,0.55,0.55,0.55,0.68
Customer City,144415.0,0.55,0.05,0.27,0.54,0.55,0.56,0.83
Customer Country,144415.0,0.55,0.00,0.55,0.55,0.55,0.55,0.55
Customer Segment,144415.0,0.55,0.00,0.55,0.55,0.55,0.55,0.55
Customer State,144415.0,0.55,0.01,0.44,0.55,0.55,0.55,0.62
Customer Zipcode,144415.0,-0.00,1.00,-0.96,-0.94,-0.45,1.13,1.68
Department Name,144415.0,0.55,0.00,0.54,0.55,0.55,0.55,0.59
Market,144415.0,0.55,0.00,0.54,0.55,0.55,0.55,0.55


In [25]:
display(X_test_lr_scaled.describe().T.round(2))

,count,mean,std,min,25%,50%,75%,max
Type,36104.0,0.55,0.04,0.49,0.49,0.57,0.57,0.58
Days for shipment (scheduled),36104.0,-0.00,1.00,-2.13,-0.68,0.78,0.78,0.78
Category Name,36104.0,0.55,0.01,0.49,0.55,0.55,0.55,0.68
Customer City,36104.0,0.55,0.05,0.27,0.54,0.55,0.56,0.83
Customer Country,36104.0,0.55,0.00,0.55,0.55,0.55,0.55,0.55
Customer Segment,36104.0,0.55,0.00,0.55,0.55,0.55,0.55,0.55
Customer State,36104.0,0.55,0.01,0.44,0.55,0.55,0.55,0.62
Customer Zipcode,36104.0,-0.00,1.00,-0.94,-0.94,-0.44,1.12,1.68
Department Name,36104.0,0.55,0.00,0.54,0.55,0.55,0.55,0.59
Market,36104.0,0.55,0.00,0.54,0.55,0.55,0.55,0.55


# Automatic Multicollinearity Reduction Using VIF

In [26]:
def iterative_vif_drop(df, threshold=10.0):
    current_df = df.copy()
    while True:
        vif_values = []
        cols = current_df.columns.tolist()
        for i in range(len(cols)):
            vif = variance_inflation_factor(current_df.values, i)
            vif_values.append(vif)

        max_idx = np.argmax(vif_values)
        if vif_values[max_idx] > threshold:
            print(f"Dropping {cols[max_idx]} with VIF {vif_values[max_idx]:.2f}")
            current_df = current_df.drop(columns=[cols[max_idx]])
        else:
            break
    return current_df

survived_numeric_df = iterative_vif_drop(X_train_lr_scaled[numeric_cols_before_encoding], threshold=10.0)
selected_vif_columns = survived_numeric_df.columns.tolist()

# Identify target-encoded columns to preserve them
target_encoded_cols = [col for col in X_train_lr.columns if col not in numeric_cols_before_encoding]

# Combine lists
final_lr_features = selected_vif_columns + target_encoded_cols

# Filter the training/testing scaled sets down to these clean features
X_train_lr_scaled = X_train_lr_scaled[final_lr_features]
X_test_lr_scaled = X_test_lr_scaled[final_lr_features]



/usr/local/lib/python3.12/dist-packages/statsmodels/stats/outliers_influence.py:197: RuntimeWarning: divide by zero encountered in scalar divide
  vif = 1. / (1. - r_squared_i)


Dropping regional_congestion_score with VIF inf
Dropping department_avg_discount with VIF 19417.59
Dropping high_value_order with VIF 1417.59
Dropping Sales with VIF 806.08
Dropping product_avg_sales with VIF 332.44
Dropping Order Item Total with VIF 33.15
Dropping region_avg_sales with VIF 19.98
Dropping order_complexity_score with VIF 16.99


# Final Validation

In [27]:
print("\nFinal Logistic Regression Training Shape:")
print(X_train_lr_scaled.shape)

print("\nFinal Logistic Regression Testing Shape:")
print(X_test_lr_scaled.shape)

print("\nFinal Tree Model Training Shape:")
print(X_train_tree.shape)

print("\nFinal Tree Model Testing Shape:")
print(X_test_tree.shape)

print("\nRemaining Missing Values In LR Training Set:")
print(X_train_lr_scaled.isnull().sum().sum())

print("\nRemaining Missing Values In LR Testing Set:")
print(X_test_lr_scaled.isnull().sum().sum())


Final Logistic Regression Training Shape:
(144415, 32)

Final Logistic Regression Testing Shape:
(36104, 32)

Final Tree Model Training Shape:
(144415, 40)

Final Tree Model Testing Shape:
(36104, 40)

Remaining Missing Values In LR Training Set:
0

Remaining Missing Values In LR Testing Set:
0


In [28]:
print("X_train_lr_scaled.columns", X_train_lr_scaled.columns)
print("X_train_tree.columns", X_train_tree.columns)

print("traing and test set has same columns?")
print(X_train_lr_scaled.columns.equals(X_test_lr_scaled.columns))
print(X_train_tree.columns.equals(X_test_tree.columns))

X_train_lr_scaled.columns Index(['Days for shipment (scheduled)', 'Customer Zipcode',
       'Order Item Discount', 'Order Item Discount Rate',
       'Order Item Product Price', 'Order Item Quantity',
       'shipping_pressure_index', 'is_high_urgency_mode', 'is_bulk_order',
       'discount_per_item', 'customer_order_count', 'customer_total_quantity',
       'customer_avg_order_value', 'customer_avg_discount',
       'product_order_frequency', 'product_avg_discount',
       'region_order_volume', 'region_avg_discount', 'department_avg_sales',
       'department_order_volume', 'Type', 'Category Name', 'Customer City',
       'Customer Country', 'Customer Segment', 'Customer State',
       'Department Name', 'Market', 'Order Country', 'Order Region',
       'Product Name', 'Shipping Mode'],
      dtype='object')
X_train_tree.columns Index(['Type', 'Days for shipment (scheduled)', 'Category Name',
       'Customer City', 'Customer Country', 'Customer Segment',
       'Customer State', '

In [29]:

local_temp_artifact_dir = Path("mlflow_artifacts")
local_temp_artifact_dir.mkdir(parents=True, exist_ok=True)

In [30]:
class LateDeliveryPreprocessingPipeline:
    def __init__(self, artifacts: dict):
        # Store mappings and defaults
        self.imputation_values = artifacts["imputation_values"]
        self.high_urgency_modes = artifacts["high_urgency_modes"]
        self.dropped_columns = artifacts["dropped_columns"]

        self.regional_volume_shares = artifacts["regional_volume_shares"]
        self.global_region_mean = artifacts["global_region_mean"]

        self.cust_counts_map = artifacts["cust_counts_map"]
        self.cust_sums_map = artifacts["cust_sums_map"]
        self.cust_sales_map = artifacts["cust_sales_map"]
        self.cust_disc_map = artifacts["cust_disc_map"]
        self.global_cust_count = artifacts["global_cust_count"]
        self.global_cust_sum = artifacts["global_cust_sum"]
        self.global_cust_sales = artifacts["global_cust_sales"]
        self.global_cust_disc = artifacts["global_cust_disc"]

        self.prod_counts_map = artifacts["prod_counts_map"]
        self.prod_sales_map = artifacts["prod_sales_map"]
        self.prod_disc_map = artifacts["prod_disc_map"]
        self.global_prod_count = artifacts["global_prod_count"]
        self.global_prod_sales = artifacts["global_prod_sales"]
        self.global_prod_disc = artifacts["global_prod_disc"]

        self.region_counts_map = artifacts["region_counts_map"]
        self.region_sales_map = artifacts["region_sales_map"]
        self.region_disc_map = artifacts["region_disc_map"]
        self.global_region_count = artifacts["global_region_count"]
        self.global_region_sales = artifacts["global_region_sales"]
        self.global_region_disc = artifacts["global_region_disc"]

        self.dept_sales_map = artifacts["dept_sales_map"]
        self.dept_counts_map = artifacts["dept_counts_map"]
        self.dept_disc_map = artifacts["dept_disc_map"]
        self.global_dept_sales = artifacts["global_dept_sales"]
        self.global_dept_count = artifacts["global_dept_count"]
        self.global_dept_disc = artifacts["global_dept_disc"]

        self.outlier_bounds = artifacts["outlier_bounds"]
        self.structural_cols = artifacts["structural_cols"]

        # Models/Transformers
        self.encoder = artifacts["encoder"]
        self.scaler = artifacts["scaler"]

        # Features lists to guarantee strict columns ordering
        self.vif_selected_columns = artifacts["vif_selected_columns"]

        self.scaling_columns = artifacts["scaling_columns"]

        self.lr_features = artifacts["selected_lr_columns"]
        self.tree_features = artifacts["tree_columns"]

    def _feature_engineering(self, df):
        df = df.copy()

        # Shipping Pressure
        df["shipping_pressure_index"] = np.where(
            df["Days for shipment (scheduled)"] == 0,
            df["Order Item Quantity"],
            df["Order Item Quantity"] / df["Days for shipment (scheduled)"]
        )

        # Flags
        df["is_high_urgency_mode"] = df["Shipping Mode"].isin(self.high_urgency_modes).astype(int)
        df["is_bulk_order"] = (df["Order Item Quantity"] > 3).astype(int)

        # Complexity & stress
        df["order_complexity_score"] = (df["Order Item Quantity"] * df["Order Item Product Price"]) * (1 - df["Order Item Discount Rate"])
        df["discount_per_item"] = df["Order Item Discount"] / (df["Order Item Quantity"] + 1)
        df["high_value_order"] = df["Order Item Product Price"] * df["Order Item Quantity"]

        # Regional congestion score fallback mapping
        df["regional_congestion_score"] = df["Order Region"].map(self.regional_volume_shares).fillna(self.global_region_mean)

        return df

    def _map_features(self, df):
        df = df.copy()

        # Customer
        df["customer_order_count"] = df["Order Customer Id"].map(self.cust_counts_map).fillna(self.global_cust_count)
        df["customer_total_quantity"] = df["Order Customer Id"].map(self.cust_sums_map).fillna(self.global_cust_sum)
        df["customer_avg_order_value"] = df["Order Customer Id"].map(self.cust_sales_map).fillna(self.global_cust_sales)
        df["customer_avg_discount"] = df["Order Customer Id"].map(self.cust_disc_map).fillna(self.global_cust_disc)

        # Product
        df["product_order_frequency"] = df["Product Name"].map(self.prod_counts_map).fillna(self.global_prod_count)
        df["product_avg_sales"] = df["Product Name"].map(self.prod_sales_map).fillna(self.global_prod_sales)
        df["product_avg_discount"] = df["Product Name"].map(self.prod_disc_map).fillna(self.global_prod_disc)

        # Region
        df["region_order_volume"] = df["Order Region"].map(self.region_counts_map).fillna(self.global_region_count)
        df["region_avg_sales"] = df["Order Region"].map(self.region_sales_map).fillna(self.global_region_sales)
        df["region_avg_discount"] = df["Order Region"].map(self.region_disc_map).fillna(self.global_region_disc)

        # Department
        df["department_avg_sales"] = df["Department Name"].map(self.dept_sales_map).fillna(self.global_dept_sales)
        df["department_order_volume"] = df["Department Name"].map(self.dept_counts_map).fillna(self.global_dept_count)
        df["department_avg_discount"] = df["Department Name"].map(self.dept_disc_map).fillna(self.global_dept_disc)

        return df

    def _clip_outliers(self, df):
        df = df.copy()
        for col, (low, high) in self.outlier_bounds.items():
            if col in df.columns:
                df[col] = df[col].clip(low, high)
        return df


    def transform(self, df):
        df = df.copy()

        df = df.drop(
            columns=self.dropped_columns,
            errors="ignore"
        )

        df = df.fillna(self.imputation_values)

        df = self._feature_engineering(df)
        df = self._map_features(df)

        df = df.drop(columns=self.structural_cols, errors="ignore")

        print("before transform shape:", df.shape)

        df = self.encoder.transform(df)

        print("after transform shape:", df.shape)

        df = self._clip_outliers(df)

        return df


    def transform_lr(self, df):
        df = self.transform(df)

        # Scale using the identical 26 features seen during training fit
        df[self.scaling_columns] = self.scaler.transform(df[self.scaling_columns])

        # Drop the VIF multi-collinear features safely here via reindexing
        df = df.reindex(columns=self.lr_features, fill_value=0)
        return df

    def transform_tree(self, df):
        df = self.transform(df)
        return df[self.tree_features]


# Create the artifact container explicitly including transformers
final_artifacts = {
    "dropped_columns": (
        high_cardinality_cols
        + future_leakage_columns
        + redundant_columns
        + ["Late_delivery_risk"]
    ),
    "imputation_values": imputation_values,
    "high_urgency_modes": high_urgency_modes,
    "regional_volume_shares": regional_volume_shares,
    "global_region_mean": global_region_mean,

    "cust_counts_map": cust_counts_map, "cust_sums_map": cust_sums_map, "cust_sales_map": cust_sales_map, "cust_disc_map": cust_disc_map,
    "global_cust_count": global_cust_count, "global_cust_sum": global_cust_sum, "global_cust_sales": global_cust_sales, "global_cust_disc": global_cust_disc,

    "prod_counts_map": prod_counts_map, "prod_sales_map": prod_sales_map, "prod_disc_map": prod_disc_map,
    "global_prod_count": global_prod_count, "global_prod_sales": global_prod_sales, "global_prod_disc": global_prod_disc,

    "region_counts_map": region_counts_map, "region_sales_map": region_sales_map, "region_disc_map": region_disc_map,
    "global_region_count": global_region_count, "global_region_sales": global_region_sales, "global_region_disc": global_region_disc,

    "dept_sales_map": dept_sales_map, "dept_counts_map": dept_counts_map, "dept_disc_map": dept_disc_map,
    "global_dept_sales": global_dept_sales, "global_dept_count": global_dept_count, "global_dept_disc": global_dept_disc,

    "outlier_bounds": outlier_bounds,
    "structural_cols": structural_cols,

    "encoder": encoder,
    "scaler": scaler,

    "vif_selected_columns": selected_vif_columns,
    "scaling_columns": scaler.feature_names_in_.tolist(),
    "selected_lr_columns": X_train_lr_scaled.columns.tolist(),
    "tree_columns": X_train_tree.columns.tolist()
}

# Instantiate and save pipeline object
pipeline_path = "late_delivery_pipeline.pkl"
pipeline = LateDeliveryPreprocessingPipeline(final_artifacts)
joblib.dump(pipeline, pipeline_path)
print(f"Pipeline serialized successfully to {pipeline_path}")


Pipeline serialized successfully to late_delivery_pipeline.pkl


In [31]:
# =====================================
# Verify Pipeline Reproduces Training Output
# =====================================

print("\nVerifying pipeline consistency...")

# Logistic Regression pipeline output
pipeline_lr_output = pipeline.transform_lr(X_train_raw.copy())

# Tree pipeline output
pipeline_tree_output = pipeline.transform_tree(X_train_raw.copy())

print("\nShape Comparison")
print("Original LR Shape:", X_train_lr_scaled.shape)
print("Pipeline LR Shape:", pipeline_lr_output.shape)

print("Original Tree Shape:", X_train_tree.shape)
print("Pipeline Tree Shape:", pipeline_tree_output.shape)

# Verify column ordering
lr_columns_match = list(X_train_lr_scaled.columns) == list(pipeline_lr_output.columns)
tree_columns_match = list(X_train_tree.columns) == list(pipeline_tree_output.columns)

print("\nColumn Order Match")
print("LR:", lr_columns_match)
print("Tree:", tree_columns_match)

# Verify actual values
lr_values_match = np.allclose(
    X_train_lr_scaled.values,
    pipeline_lr_output.values,
    rtol=1e-6,
    atol=1e-6
)

tree_values_match = np.allclose(
    X_train_tree.values,
    pipeline_tree_output.values,
    rtol=1e-6,
    atol=1e-6
)

print("\nValue Match")
print("LR:", lr_values_match)
print("Tree:", tree_values_match)

# Detailed diagnostics if mismatch exists
if not lr_values_match:

    diff_mask = ~np.isclose(
        X_train_lr_scaled.values,
        pipeline_lr_output.values,
        rtol=1e-6,
        atol=1e-6
    )

    mismatch_count = diff_mask.sum()

    print(f"\nLR mismatches found: {mismatch_count}")

    mismatch_locations = np.argwhere(diff_mask)

    for row_idx, col_idx in mismatch_locations[:10]:

        col_name = X_train_lr_scaled.columns[col_idx]

        print(
            f"Row={row_idx}, "
            f"Column={col_name}, "
            f"Expected={X_train_lr_scaled.iloc[row_idx, col_idx]}, "
            f"Pipeline={pipeline_lr_output.iloc[row_idx, col_idx]}"
        )

if not tree_values_match:

    diff_mask = ~np.isclose(
        X_train_tree.values,
        pipeline_tree_output.values,
        rtol=1e-6,
        atol=1e-6
    )

    mismatch_count = diff_mask.sum()

    print(f"\nTree mismatches found: {mismatch_count}")

    mismatch_locations = np.argwhere(diff_mask)

    for row_idx, col_idx in mismatch_locations[:10]:

        col_name = X_train_tree.columns[col_idx]

        print(
            f"Row={row_idx}, "
            f"Column={col_name}, "
            f"Expected={X_train_tree.iloc[row_idx, col_idx]}, "
            f"Pipeline={pipeline_tree_output.iloc[row_idx, col_idx]}"
        )

if (
    lr_columns_match
    and tree_columns_match
    and lr_values_match
    and tree_values_match
):
    print("\nSUCCESS: Pipeline output exactly matches training preprocessing.")
else:
    raise ValueError(
        "Pipeline verification failed. Training preprocessing and deployment pipeline do not match."
    )


Verifying pipeline consistency...
before transform shape: (144415, 40)
after transform shape: (144415, 40)
before transform shape: (144415, 40)
after transform shape: (144415, 40)

Shape Comparison
Original LR Shape: (144415, 32)
Pipeline LR Shape: (144415, 32)
Original Tree Shape: (144415, 40)
Pipeline Tree Shape: (144415, 40)

Column Order Match
LR: True
Tree: True

Value Match
LR: True
Tree: True

SUCCESS: Pipeline output exactly matches training preprocessing.


In [32]:

# =====================================
# Ensure No Active Run
# =====================================
if mlflow.active_run():
    mlflow.end_run()

# =====================================
# Start MLflow Run
# =====================================
with mlflow.start_run(run_name="data_preparation_pipeline"):

    mlflow.set_tags({
        "project": "late_delivery_prediction",
        "pipeline_stage": "preprocessing"
    })

    # =====================================
    # Log Parameters
    # =====================================
    mlflow.log_param("original_rows", raw_df.shape[0])
    mlflow.log_param("original_columns", raw_df.shape[1])
    mlflow.log_param("final_lr_features", X_train_lr_scaled.shape[1])
    mlflow.log_param("final_tree_features", X_train_tree.shape[1])

    # =====================================
    # Save Dataset Artifacts Safely (String Columns Rule)
    # =====================================
    def safe_to_parquet(df, path):
        df_copy = df.copy()
        df_copy.columns = df_copy.columns.astype(str)
        df_copy.to_parquet(path, index=False)

    safe_to_parquet(X_train_lr_scaled, local_temp_artifact_dir / "X_train_lr_scaled.parquet")
    safe_to_parquet(X_test_lr_scaled, local_temp_artifact_dir / "X_test_lr_scaled.parquet")
    safe_to_parquet(X_train_tree, local_temp_artifact_dir / "X_train_tree.parquet")
    safe_to_parquet(X_test_tree, local_temp_artifact_dir / "X_test_tree.parquet")
    safe_to_parquet(y_train.to_frame(), local_temp_artifact_dir / "y_train.parquet")
    safe_to_parquet(y_test.to_frame(), local_temp_artifact_dir / "y_test.parquet")

    # =====================================
    # Save Pipeline Objects
    # =====================================
    joblib.dump(categorical_cols, local_temp_artifact_dir / "categorical_cols.pkl")
    joblib.dump(scaler, local_temp_artifact_dir / "scaler.joblib")
    joblib.dump(encoder, local_temp_artifact_dir / "target_encoder.joblib")
    joblib.dump(X_train_lr_scaled.columns.tolist(), local_temp_artifact_dir / "selected_lr_columns.pkl")

    # =====================================
    # Log Artifacts Flat
    # =====================================
    for artifact_file in local_temp_artifact_dir.iterdir():
        if artifact_file.is_file():
            mlflow.log_artifact(str(artifact_file.resolve()))

    # Log deployment object directly
    mlflow.log_artifact(pipeline_path)

    script_name = "late_delivery_preprocessing_pipeline.ipynb"
    if os.path.exists(script_name):
        mlflow.log_artifact(script_name)


# =====================================
# Cleanup Temporary Artifact Directory
# =====================================
try:
    shutil.rmtree(local_temp_artifact_dir)
    print(f"\nCleaned up temporary artifact directory:\n{local_temp_artifact_dir}")
except Exception as e:
    print(f"\nTemporary cleanup skipped: {e}")

print("\nMLflow preprocessing tracking completed successfully.")

# =====================================
# Verify Latest Run And Artifacts
# =====================================
client = mlflow.tracking.MlflowClient()
preprocessing_exp = mlflow.get_experiment_by_name(experiment_name)

runs = mlflow.search_runs(
    experiment_ids=[preprocessing_exp.experiment_id],
    order_by=["attributes.start_time DESC"],
    max_results=1
)

latest_run_id = runs.iloc[0]["run_id"]
print(f"\nLatest Run ID: {latest_run_id}")

artifacts = client.list_artifacts(latest_run_id)
print("\nArtifacts Archive:")
for artifact in artifacts:
    print(artifact.path)

🏃 View run data_preparation_pipeline at: https://dagshub.com/sharifa-mohamed/late_delivery_prediction.mlflow/#/experiments/1/runs/8dc08406d088481b9202d287c40d222d
🧪 View experiment at: https://dagshub.com/sharifa-mohamed/late_delivery_prediction.mlflow/#/experiments/1

Cleaned up temporary artifact directory:
mlflow_artifacts

MLflow preprocessing tracking completed successfully.

Latest Run ID: 8dc08406d088481b9202d287c40d222d

Artifacts Archive:
X_test_lr_scaled.parquet
X_test_tree.parquet
X_train_lr_scaled.parquet
X_train_tree.parquet
categorical_cols.pkl
late_delivery_pipeline.pkl
late_delivery_preprocessing_pipeline.ipynb
scaler.joblib
selected_lr_columns.pkl
target_encoder.joblib
y_test.parquet
y_train.parquet
